# Incremental Load Control Testing

#### This notebook validates batch registration, pipeline state changes and watermark updates before the same logic is moved into production Python.

In [2]:
# Import os so we can inspect the environment variables currently loaded.
import os

# Import the dotenv loader.
from dotenv import load_dotenv

# Reload variables from the project .env file.
load_dotenv(override=True)

# Display only non-secret connection settings.
print("Host:", os.getenv("NETFLIX_DB_HOST"))
print("Port:", os.getenv("NETFLIX_DB_PORT"))
print("Database:", os.getenv("NETFLIX_DB_NAME"))
print("ETL user:", os.getenv("NETFLIX_ETL_USER"))

Host: 127.0.0.1
Port: 5432
Database: postgres
ETL user: netflix_etl


## Test 1: Connect as the ETL user

Confirm that the notebook connects to PostgreSQL using the dedicated `netflix_etl` account before manipulating pipeline metadata.

In [4]:
# Import os for reading environment variables.
import os

# Import psycopg for PostgreSQL connections.
import psycopg

# Import load_dotenv so variables can be loaded from .env.
from dotenv import load_dotenv

# Reload the .env file and override any stale notebook environment values.
load_dotenv(override=True)

# Read the PostgreSQL host.
db_host = os.getenv("NETFLIX_DB_HOST")

# Read the PostgreSQL port.
db_port = os.getenv("NETFLIX_DB_PORT")

# Read the PostgreSQL database name.
db_name = os.getenv("NETFLIX_DB_NAME")

# Read the dedicated ETL account.
db_user = os.getenv("NETFLIX_ETL_USER")

# Read the ETL password without printing it.
db_password = os.getenv("NETFLIX_ETL_PASSWORD")

# Fail immediately if an important setting is missing.
required_values = {
    "NETFLIX_DB_HOST": db_host,
    "NETFLIX_DB_PORT": db_port,
    "NETFLIX_DB_NAME": db_name,
    "NETFLIX_ETL_USER": db_user,
    "NETFLIX_ETL_PASSWORD": db_password,
}

# Identify any missing variables.
missing_values = [
    name
    for name, value in required_values.items()
    if not value
]

# Stop execution before making a bad database connection.
if missing_values:
    raise ValueError(
        f"Missing environment variables: {missing_values}"
    )

# Open the PostgreSQL connection with the ETL identity.
connection = psycopg.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
)

# Open a cursor so SQL can be executed.
with connection.cursor() as cursor:

    # Confirm the database and PostgreSQL role actually in use.
    cursor.execute(
        """
        SELECT
            current_database(),
            current_user;
        """
    )

    # Retrieve the returned row.
    database_name, database_user = cursor.fetchone()

# Display safe verification information.
print("Connection status: SUCCESS")
print("Database:", database_name)
print("Connected user:", database_user)

Connection status: SUCCESS
Database: postgres
Connected user: netflix_etl


## Test 2: Initialise the pipeline

Create the first control record for the Netflix incremental pipeline.

This record acts as the pipeline's checkpoint and will later store the last successful watermark, last processed file and current pipeline status.

In [5]:
# Import pandas so database results can be displayed clearly.
import pandas as pd


# Give the pipeline a permanent identifier.
pipeline_name = "netflix_incremental_pipeline"


# Open a PostgreSQL cursor so we can execute SQL.
with connection.cursor() as cursor:

    # Create the pipeline control record if it does not already exist.
    cursor.execute(
        """
        INSERT INTO etl.pipeline_control (
            pipeline_name
        )
        VALUES (%s)
        ON CONFLICT (pipeline_name) DO NOTHING;
        """,
        (pipeline_name,),
    )


# Commit the transaction so the new record is permanently saved.
connection.commit()


# Read the control record back from PostgreSQL.
pipeline_state = pd.read_sql_query(
    """
    SELECT *
    FROM etl.pipeline_control
    WHERE pipeline_name = %s;
    """,
    connection,
    params=(pipeline_name,),
)


# Display the pipeline state.
display(pipeline_state)

/var/folders/12/l5ffmnws69ldbsdp_bmmh13c0000gn/T/ipykernel_39984/4206046250.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pipeline_state = pd.read_sql_query(


,pipeline_name,last_successful_load,last_watermark,last_file_name,rows_processed,status,updated_at
0,netflix_incremental_pipeline,None,None,None,0,NEVER_RUN,2026-08-18 15:19:10.784116+00:00


## Test 3: Register an incoming batch

Simulate a new Netflix data file arriving and record it before processing begins.

The batch history table gives us an audit trail for every incoming file.

In [6]:
# Create a fake filename representing an incoming batch.
test_file_name = "titles_20260818_0600.csv"


# Use a temporary checksum while testing the workflow.
# Later this value will be calculated automatically from the real file.
test_checksum = "TEST_CHECKSUM_001"


# Simulate how many records arrived in the batch.
test_rows_received = 250


# Open a PostgreSQL cursor for the batch insert.
with connection.cursor() as cursor:

    # Register the new batch before any transformation begins.
    cursor.execute(
        """
        INSERT INTO etl.batch_history (
            pipeline_name,
            file_name,
            file_checksum,
            rows_received,
            status
        )
        VALUES (%s, %s, %s, %s, 'RECEIVED')
        RETURNING batch_id;
        """,
        (
            pipeline_name,
            test_file_name,
            test_checksum,
            test_rows_received,
        ),
    )

    # Retrieve the unique batch ID generated by PostgreSQL.
    batch_id = cursor.fetchone()[0]


# Save the new batch record.
connection.commit()


# Display the batch identifier.
print("Registered batch ID:", batch_id)

Registered batch ID: 1


In [7]:
# Retrieve the batch we just registered.
registered_batch = pd.read_sql_query(
    """
    SELECT *
    FROM etl.batch_history
    WHERE batch_id = %s;
    """,
    connection,
    params=(batch_id,),
)


# Display the registered batch.
display(registered_batch)

/var/folders/12/l5ffmnws69ldbsdp_bmmh13c0000gn/T/ipykernel_39984/1773917316.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  registered_batch = pd.read_sql_query(


,batch_id,pipeline_name,file_name,file_checksum,received_at,processing_started_at,processing_completed_at,rows_received,rows_processed,status,error_message
0,1,netflix_incremental_pipeline,titles_20260818_0600.csv,TEST_CHECKSUM_001,2026-08-18 15:33:16.488712+00:00,None,None,250,None,RECEIVED,None


## Test 4: Start processing

Change both the batch and the overall pipeline state to RUNNING.

In [8]:
# Open a cursor so both status updates can be performed together.
with connection.cursor() as cursor:

    # Mark the individual batch as actively processing.
    cursor.execute(
        """
        UPDATE etl.batch_history
        SET
            status = 'RUNNING',
            processing_started_at = CURRENT_TIMESTAMP
        WHERE batch_id = %s;
        """,
        (batch_id,),
    )

    # Mark the overall pipeline as actively running.
    cursor.execute(
        """
        UPDATE etl.pipeline_control
        SET
            status = 'RUNNING',
            updated_at = CURRENT_TIMESTAMP
        WHERE pipeline_name = %s;
        """,
        (pipeline_name,),
    )


# Commit both updates.
connection.commit()

In [9]:
# Inspect the current states of both control tables.
current_pipeline_state = pd.read_sql_query(
    """
    SELECT *
    FROM etl.pipeline_control
    WHERE pipeline_name = %s;
    """,
    connection,
    params=(pipeline_name,),
)

current_batch_state = pd.read_sql_query(
    """
    SELECT *
    FROM etl.batch_history
    WHERE batch_id = %s;
    """,
    connection,
    params=(batch_id,),
)


# Display the current pipeline and batch states.
display(current_pipeline_state)
display(current_batch_state)

/var/folders/12/l5ffmnws69ldbsdp_bmmh13c0000gn/T/ipykernel_39984/1977431691.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  current_pipeline_state = pd.read_sql_query(
/var/folders/12/l5ffmnws69ldbsdp_bmmh13c0000gn/T/ipykernel_39984/1977431691.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  current_batch_state = pd.read_sql_query(


,pipeline_name,last_successful_load,last_watermark,last_file_name,rows_processed,status,updated_at
0,netflix_incremental_pipeline,None,None,None,0,RUNNING,2026-08-18 15:38:52.313536+00:00


,batch_id,pipeline_name,file_name,file_checksum,received_at,processing_started_at,processing_completed_at,rows_received,rows_processed,status,error_message
0,1,netflix_incremental_pipeline,titles_20260818_0600.csv,TEST_CHECKSUM_001,2026-08-18 15:33:16.488712+00:00,2026-08-18 15:38:52.313536+00:00,None,250,None,RUNNING,None


## Test 5: Complete a successful batch

Simulate successful processing of the current batch.

The pipeline watermark must advance only after the batch reaches SUCCESS.

In [10]:
# Import datetime so we can simulate the timestamp of the newest source record.
from datetime import datetime, timezone


# Simulate the newest source timestamp contained in this batch.
new_watermark = datetime(
    2026,
    8,
    18,
    6,
    0,
    tzinfo=timezone.utc,
)


# Simulate that every incoming row was processed successfully.
rows_processed = test_rows_received


# Open a PostgreSQL cursor so we can complete the batch.
with connection.cursor() as cursor:

    # Mark the individual batch as successfully completed.
    cursor.execute(
        """
        UPDATE etl.batch_history
        SET
            status = 'SUCCESS',
            rows_processed = %s,
            processing_completed_at = CURRENT_TIMESTAMP
        WHERE batch_id = %s;
        """,
        (
            rows_processed,
            batch_id,
        ),
    )

    # Advance the overall pipeline checkpoint only after the batch succeeds.
    cursor.execute(
        """
        UPDATE etl.pipeline_control
        SET
            last_successful_load = CURRENT_TIMESTAMP,
            last_watermark = %s,
            last_file_name = %s,
            rows_processed = %s,
            status = 'SUCCESS',
            updated_at = CURRENT_TIMESTAMP
        WHERE pipeline_name = %s;
        """,
        (
            new_watermark,
            test_file_name,
            rows_processed,
            pipeline_name,
        ),
    )


# Commit both successful state changes together.
connection.commit()

In [11]:
# Read the current pipeline control record.
successful_pipeline_state = pd.read_sql_query(
    """
    SELECT
        pipeline_name,
        last_successful_load,
        last_watermark,
        last_file_name,
        rows_processed,
        status
    FROM etl.pipeline_control
    WHERE pipeline_name = %s;
    """,
    connection,
    params=(pipeline_name,),
)


# Read the completed batch record.
successful_batch_state = pd.read_sql_query(
    """
    SELECT
        batch_id,
        file_name,
        received_at,
        processing_started_at,
        processing_completed_at,
        rows_received,
        rows_processed,
        status
    FROM etl.batch_history
    WHERE batch_id = %s;
    """,
    connection,
    params=(batch_id,),
)


# Display the pipeline result.
display(successful_pipeline_state)


# Display the batch result.
display(successful_batch_state)

/var/folders/12/l5ffmnws69ldbsdp_bmmh13c0000gn/T/ipykernel_39984/2175619085.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  successful_pipeline_state = pd.read_sql_query(
/var/folders/12/l5ffmnws69ldbsdp_bmmh13c0000gn/T/ipykernel_39984/2175619085.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  successful_batch_state = pd.read_sql_query(


,pipeline_name,last_successful_load,last_watermark,last_file_name,rows_processed,status
0,netflix_incremental_pipeline,2026-08-18 15:43:31.231873+00:00,2026-08-18 06:00:00+00:00,titles_20260818_0600.csv,250,SUCCESS


,batch_id,file_name,received_at,processing_started_at,processing_completed_at,rows_received,rows_processed,status
0,1,titles_20260818_0600.csv,2026-08-18 15:33:16.488712+00:00,2026-08-18 15:38:52.313536+00:00,2026-08-18 15:43:31.231873+00:00,250,250,SUCCESS


## Test 6: Failed batch and watermark protection

Simulate a second incoming batch that fails during processing.

The previous successful watermark must remain unchanged.

In [12]:
# Read the watermark before starting the failure test.
with connection.cursor() as cursor:

    # Retrieve the currently successful watermark.
    cursor.execute(
        """
        SELECT last_watermark
        FROM etl.pipeline_control
        WHERE pipeline_name = %s;
        """,
        (pipeline_name,),
    )

    # Store the watermark so we can compare it after the failure.
    watermark_before_failure = cursor.fetchone()[0]


# Display the protected watermark.
print("Watermark before failure:", watermark_before_failure)

Watermark before failure: 2026-08-18 07:00:00+01:00


## Now register another batch

In [13]:
# Create a second simulated incoming file.
failed_file_name = "titles_20260819_0600.csv"


# Give the second file a different test checksum.
failed_checksum = "TEST_CHECKSUM_002"


# Simulate the number of rows contained in the second batch.
failed_rows_received = 300


# Open a cursor so the second batch can be registered.
with connection.cursor() as cursor:

    # Insert the second batch as RECEIVED.
    cursor.execute(
        """
        INSERT INTO etl.batch_history (
            pipeline_name,
            file_name,
            file_checksum,
            rows_received,
            status
        )
        VALUES (%s, %s, %s, %s, 'RECEIVED')
        RETURNING batch_id;
        """,
        (
            pipeline_name,
            failed_file_name,
            failed_checksum,
            failed_rows_received,
        ),
    )

    # Capture the newly generated batch ID.
    failed_batch_id = cursor.fetchone()[0]


# Save the batch registration.
connection.commit()


# Display the new batch identifier.
print("Failed-test batch ID:", failed_batch_id)

Failed-test batch ID: 2


## Make it running

In [14]:
# Open a cursor to simulate the beginning of processing.
with connection.cursor() as cursor:

    # Mark the test batch as RUNNING.
    cursor.execute(
        """
        UPDATE etl.batch_history
        SET
            status = 'RUNNING',
            processing_started_at = CURRENT_TIMESTAMP
        WHERE batch_id = %s;
        """,
        (failed_batch_id,),
    )

    # Mark the overall pipeline as RUNNING.
    cursor.execute(
        """
        UPDATE etl.pipeline_control
        SET
            status = 'RUNNING',
            updated_at = CURRENT_TIMESTAMP
        WHERE pipeline_name = %s;
        """,
        (pipeline_name,),
    )


# Persist the RUNNING states.
connection.commit()

## delibrately fail the simulation

In [15]:
# Create a fake error message representing a transformation failure.
simulated_error = "Simulated transformation failure during incremental load."


# Open a cursor so the failed state can be recorded.
with connection.cursor() as cursor:

    # Mark the individual batch as FAILED.
    cursor.execute(
        """
        UPDATE etl.batch_history
        SET
            status = 'FAILED',
            processing_completed_at = CURRENT_TIMESTAMP,
            error_message = %s
        WHERE batch_id = %s;
        """,
        (
            simulated_error,
            failed_batch_id,
        ),
    )

    # Mark the overall pipeline as FAILED.
    # Notice that last_watermark is deliberately NOT updated.
    cursor.execute(
        """
        UPDATE etl.pipeline_control
        SET
            status = 'FAILED',
            updated_at = CURRENT_TIMESTAMP
        WHERE pipeline_name = %s;
        """,
        (pipeline_name,),
    )


# Save the failure information.
connection.commit()

## Perform critical check

In [16]:
# Read the watermark after the simulated failure.
with connection.cursor() as cursor:

    # Retrieve the pipeline watermark and status.
    cursor.execute(
        """
        SELECT
            last_watermark,
            status
        FROM etl.pipeline_control
        WHERE pipeline_name = %s;
        """,
        (pipeline_name,),
    )

    # Store the returned values.
    watermark_after_failure, pipeline_status = cursor.fetchone()


# Display the comparison.
print("Watermark before failure:", watermark_before_failure)
print("Watermark after failure: ", watermark_after_failure)
print("Pipeline status:         ", pipeline_status)


# Confirm programmatically that the watermark did not move.
assert watermark_after_failure == watermark_before_failure


# Confirm that the pipeline correctly reports failure.
assert pipeline_status == "FAILED"


# Print confirmation when both tests pass.
print("PASS: failed batch did not advance the watermark.")

Watermark before failure: 2026-08-18 07:00:00+01:00
Watermark after failure:  2026-08-18 07:00:00+01:00
Pipeline status:          FAILED
PASS: failed batch did not advance the watermark.


## Test 7: Duplicate batch detection


Verify that an already successfully processed file can be identified before it enters the pipeline again.

In [17]:
# Use the checksum from our already successful first batch.
incoming_checksum = test_checksum


# Open a cursor to check whether this source has already succeeded.
with connection.cursor() as cursor:

    # Search for a successful batch with the same checksum.
    cursor.execute(
        """
        SELECT
            batch_id,
            file_name,
            status
        FROM etl.batch_history
        WHERE file_checksum = %s
          AND status = 'SUCCESS'
        LIMIT 1;
        """,
        (incoming_checksum,),
    )

    # Retrieve the matching batch when one exists.
    duplicate_batch = cursor.fetchone()


# Determine whether the incoming file is a duplicate.
is_duplicate = duplicate_batch is not None


# Display the result.
print("Duplicate detected:", is_duplicate)


# Display information about the original successful batch.
print("Original batch:", duplicate_batch)


# Confirm that duplicate detection works.
assert is_duplicate is True


# Print confirmation when the test succeeds.
print("PASS: previously processed batch was detected.")

Duplicate detected: True
Original batch: (1, 'titles_20260818_0600.csv', 'SUCCESS')
PASS: previously processed batch was detected.


## Test 8: Late-arriving records

Validate the incremental lookback strategy.

The pipeline intentionally re-reads a small period before the last successful watermark so records that arrive late are not missed.

In [18]:
# Import timedelta so we can subtract a lookback period from the watermark.
from datetime import timedelta


# Read the most recent successful watermark from PostgreSQL.
with connection.cursor() as cursor:

    # Retrieve only the last successfully processed source timestamp.
    cursor.execute(
        """
        SELECT last_watermark
        FROM etl.pipeline_control
        WHERE pipeline_name = %s;
        """,
        (pipeline_name,),
    )

    # Store the watermark returned by PostgreSQL.
    current_watermark = cursor.fetchone()[0]


# Define how far backwards each incremental load should re-read.
lookback_minutes = 10


# Create the true extraction starting point.
extract_from = current_watermark - timedelta(minutes=lookback_minutes)


# Display the calculated incremental window.
print("Current watermark:", current_watermark)
print("Lookback period:", lookback_minutes, "minutes")
print("Next extraction starts from:", extract_from)

Current watermark: 2026-08-18 07:00:00+01:00
Lookback period: 10 minutes
Next extraction starts from: 2026-08-18 06:50:00+01:00


## simulate a late record

In [19]:
# Simulate source records received during the next incremental batch.
simulated_records = [
    {"title_id": 101, "source_timestamp": current_watermark + timedelta(minutes=2)},
    {"title_id": 102, "source_timestamp": current_watermark + timedelta(minutes=5)},
    {"title_id": 103, "source_timestamp": current_watermark - timedelta(minutes=3)},
]


# Display the simulated records.
for record in simulated_records:
    print(record)

{'title_id': 101, 'source_timestamp': datetime.datetime(2026, 8, 18, 7, 2, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos'))}
{'title_id': 102, 'source_timestamp': datetime.datetime(2026, 8, 18, 7, 5, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos'))}
{'title_id': 103, 'source_timestamp': datetime.datetime(2026, 8, 18, 6, 57, tzinfo=zoneinfo.ZoneInfo(key='Africa/Lagos'))}


## filter using our extraction boundary

In [20]:
# Keep every record that falls inside the incremental extraction window.
records_selected = [
    record
    for record in simulated_records
    if record["source_timestamp"] >= extract_from
]


# Display the records the next load would capture.
for record in records_selected:
    print(
        "Selected:",
        record["title_id"],
        record["source_timestamp"],
    )


# Confirm that the late 05:57 record was recovered.
selected_ids = [record["title_id"] for record in records_selected]


# The late-arriving record must be included.
assert 103 in selected_ids


# Display confirmation when the late record is successfully recovered.
print("PASS: late-arriving record was captured by the lookback window.")

Selected: 101 2026-08-18 07:02:00+01:00
Selected: 102 2026-08-18 07:05:00+01:00
Selected: 103 2026-08-18 06:57:00+01:00
PASS: late-arriving record was captured by the lookback window.


## Test 9: Late batch arrival

Simulate a scheduled pipeline waiting for an expected source file rather than failing immediately when the file arrives slightly late

In [21]:
# Import datetime utilities for the simulated schedule.
from datetime import datetime, timezone, timedelta


# Simulate the scheduled Airflow execution time.
scheduled_time = datetime(
    2026,
    8,
    18,
    9,
    0,
    tzinfo=timezone.utc,
)


# Simulate the source file arriving two minutes late.
actual_file_arrival = scheduled_time + timedelta(minutes=2)


# Define how long the pipeline is willing to wait.
allowed_wait_minutes = 10


# Calculate the latest acceptable arrival time.
arrival_deadline = scheduled_time + timedelta(minutes=allowed_wait_minutes)


# Check whether the source arrived within the permitted window.
file_arrived_within_sla = actual_file_arrival <= arrival_deadline


# Display the simulated timing.
print("Scheduled run:", scheduled_time)
print("File arrival:", actual_file_arrival)
print("Deadline:", arrival_deadline)
print("Within SLA:", file_arrived_within_sla)


# Confirm that a file arriving two minutes late is still acceptable.
assert file_arrived_within_sla is True


# Display confirmation.
print("PASS: slightly late batch would still be processed.")

Scheduled run: 2026-08-18 09:00:00+00:00
File arrival: 2026-08-18 09:02:00+00:00
Deadline: 2026-08-18 09:10:00+00:00
Within SLA: True
PASS: slightly late batch would still be processed.


## Test a genuinely missing batch

In [22]:
# Simulate a file that arrives too late for the configured SLA.
very_late_file_arrival = scheduled_time + timedelta(minutes=30)


# Determine whether this arrival is still acceptable.
very_late_within_sla = very_late_file_arrival <= arrival_deadline


# Display the result.
print("Very late arrival:", very_late_file_arrival)
print("Within SLA:", very_late_within_sla)


# Confirm that the pipeline would reject/timeout this arrival.
assert very_late_within_sla is False


# Display confirmation.
print("PASS: excessively late batch would trigger the timeout path.")

Very late arrival: 2026-08-18 09:30:00+00:00
Within SLA: False
PASS: excessively late batch would trigger the timeout path.
